Weapons and Ammunition data analysis


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

pd.options.display.max_columns = None

df = pd.read_csv('/workspaces/amc-research-sprint-lh_gl/duplicates_result.csv', encoding='latin-1')
weapons_df = df[df['item_type'] == 3]

### Select weapons life cycle stages

In [9]:
cols_to_keep = [
    # Identifiers
    'item_type', 'item',

    # Lifecycle stages - ban flags
    'ban_development',
    'ban_testing',
    'ban_production',
    'ban_acquisition',
    'ban_possession',
    'ban_station',
    'ban_transfer',
    'ban_use',
    'ban_disposal',

    # Lifecycle stages - restriction flags
    'restriction_development',
    'testing_restriction',       # note: inconsistent naming in the dataset
    'restriction_production',
    'restriction_acquisition',
    'restriction_possession',
    'restriction_transfer',
    'restriction_use',
    'restriction_disposal'
]

weapons_lifecycle_df = weapons_df[cols_to_keep]


### Testing correlations between different life cycle stages of weapons/ammunitions

In [10]:
# Separate ban and restriction columns
ban_cols = [c for c in weapons_lifecycle_df.columns if c.startswith('ban_')]
restriction_cols = [c for c in weapons_lifecycle_df.columns if 'restriction' in c]

# Correlation between every ban col vs every restriction col
corr_matrix = weapons_lifecycle_df[ban_cols + restriction_cols].corr()

# Slice to only show ban vs restriction (not ban vs ban or restriction vs restriction)
corr_ban_vs_restriction = corr_matrix.loc[ban_cols, restriction_cols]
print(corr_ban_vs_restriction)

                 restriction_development  testing_restriction  \
ban_development                      NaN                  NaN   
ban_testing                          NaN                  NaN   
ban_production                 -0.091287            -0.091287   
ban_acquisition                -0.091287            -0.091287   
ban_possession                       NaN                  NaN   
ban_station                          NaN                  NaN   
ban_transfer                   -0.169031            -0.169031   
ban_use                        -0.091287            -0.091287   
ban_disposal                   -0.133333            -0.133333   

                 restriction_production  restriction_acquisition  \
ban_development                     NaN                      NaN   
ban_testing                         NaN                      NaN   
ban_production                -0.209165                -0.184637   
ban_acquisition               -0.209165                -0.184637   
ban_posse

In [11]:
# Stack matrix into a series and sort
corr_ranked = (
    corr_ban_vs_restriction
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
    .sort_values('correlation', ascending=False)
)

print(corr_ranked.head(40))

                ban              restriction  correlation
70     ban_disposal          restriction_use     0.164992
68     ban_disposal   restriction_possession     0.065465
66     ban_disposal   restriction_production     0.065465
69     ban_disposal     restriction_transfer    -0.021517
23   ban_production     restriction_disposal    -0.062500
31  ban_acquisition     restriction_disposal    -0.062500
63          ban_use     restriction_disposal    -0.062500
16   ban_production  restriction_development    -0.091287
56          ban_use  restriction_development    -0.091287
57          ban_use      testing_restriction    -0.091287
25  ban_acquisition      testing_restriction    -0.091287
24  ban_acquisition  restriction_development    -0.091287
17   ban_production      testing_restriction    -0.091287
71     ban_disposal     restriction_disposal    -0.091287
55     ban_transfer     restriction_disposal    -0.115728
64     ban_disposal  restriction_development    -0.133333
65     ban_dis

In [12]:
# Pearson - default, fine for binary
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='pearson')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_acquisition,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,-0.115728,-0.115728,NaN,NaN,1.000000,-0.115728,-0.169031,-0.169031,-0.169031,-0.387298,-0.341882,-0.387298,-0.490990,-0.298807,-0.115728
ban_use,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_disposal,NaN,NaN,-0.091287,-0.091287,NaN,NaN,-0.169031,-0.091287,1.000000,-0.133333,-0.133333,0.065465,-0.269680,0.065465,-0.021517,0.164992,-0.091287
restriction_development,NaN,NaN,-0.091287,-0.091287,NaN,NaN,-0.169031,-0.091287,-0.133333,1.000000,1.000000,0.436436,-0.269680,0.436436,0.344265,-0.235702,-0.091287


In [13]:
# Spearman - better for ordinal/binary data, more robust
weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

,ban_development,ban_testing,ban_production,ban_acquisition,ban_possession,ban_station,ban_transfer,ban_use,ban_disposal,restriction_development,testing_restriction,restriction_production,restriction_acquisition,restriction_possession,restriction_transfer,restriction_use,restriction_disposal
ban_development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_testing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_production,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_acquisition,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_station,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ban_transfer,NaN,NaN,-0.115728,-0.115728,NaN,NaN,1.000000,-0.115728,-0.169031,-0.169031,-0.169031,-0.387298,-0.341882,-0.387298,-0.490990,-0.298807,-0.115728
ban_use,NaN,NaN,1.000000,1.000000,NaN,NaN,-0.115728,1.000000,-0.091287,-0.091287,-0.091287,-0.209165,-0.184637,-0.209165,-0.265165,-0.161374,-0.062500
ban_disposal,NaN,NaN,-0.091287,-0.091287,NaN,NaN,-0.169031,-0.091287,1.000000,-0.133333,-0.133333,0.065465,-0.269680,0.065465,-0.021517,0.164992,-0.091287
restriction_development,NaN,NaN,-0.091287,-0.091287,NaN,NaN,-0.169031,-0.091287,-0.133333,1.000000,1.000000,0.436436,-0.269680,0.436436,0.344265,-0.235702,-0.091287


Filter strong correlations

In [14]:
corr = weapons_lifecycle_df[ban_cols + restriction_cols].corr(method='spearman')

strong_corr = (
    corr
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'ban', 'level_1': 'restriction', 0: 'correlation'})
)

strong_corr = strong_corr[
    ((strong_corr['correlation'] > 0.5) | (strong_corr['correlation'] < -0.5)) & (strong_corr['correlation'] != 1.0)
].sort_values('correlation', ascending=False)

print(strong_corr)

                        ban             restriction  correlation
251    restriction_transfer  restriction_possession     0.788811
235  restriction_possession    restriction_transfer     0.788811
268         restriction_use  restriction_possession     0.771517
236  restriction_possession         restriction_use     0.771517
232  restriction_possession  restriction_production     0.757143
200  restriction_production  restriction_possession     0.757143
269         restriction_use    restriction_transfer     0.608581
253    restriction_transfer         restriction_use     0.608581
249    restriction_transfer  restriction_production     0.549350
201  restriction_production    restriction_transfer     0.549350
202  restriction_production         restriction_use     0.509201
266         restriction_use  restriction_production     0.509201
